In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

BASE = Path(r"C:\Users\npd20\Downloads\ĐACN_v2")
OUT  = BASE / "data" / "processed" / "synthetic_cells"
SPLITS = BASE / "data" / "splits"
SPLITS.mkdir(parents=True, exist_ok=True)

# Collect tất cả ảnh + label
records = []
for score_dir in sorted(OUT.iterdir()):
    if not score_dir.is_dir():
        continue
    label = score_dir.name.replace("_", ".")  # "6_8" → "6.8"
    for img_path in score_dir.glob("*.png"):
        records.append({
            "image_path": str(img_path),
            "label": label
        })

df = pd.DataFrame(records)
print(f"Total: {len(df):,} images | {df['label'].nunique()} classes")
print(df['label'].value_counts().sort_index())

Total: 205,000 images | 41 classes
label
0.0    5000
0.3    5000
0.5    5000
0.8    5000
1.0    5000
1.3    5000
1.5    5000
1.8    5000
10     5000
2.0    5000
2.3    5000
2.5    5000
2.8    5000
3.0    5000
3.3    5000
3.5    5000
3.8    5000
4.0    5000
4.3    5000
4.5    5000
4.8    5000
5.0    5000
5.3    5000
5.5    5000
5.8    5000
6.0    5000
6.3    5000
6.5    5000
6.8    5000
7.0    5000
7.3    5000
7.5    5000
7.8    5000
8.0    5000
8.3    5000
8.5    5000
8.8    5000
9.0    5000
9.3    5000
9.5    5000
9.8    5000
Name: count, dtype: int64


# Perfectly balanced. Cell 2 – Split:

In [2]:
from sklearn.model_selection import train_test_split

# Split stratified theo label
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['label']
)

# Save
train_df.to_csv(SPLITS / "train.csv", index=False)
val_df.to_csv(SPLITS / "val.csv",   index=False)
test_df.to_csv(SPLITS / "test.csv", index=False)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"\nClass distribution check (train):")
print(train_df['label'].value_counts().sort_index().to_string())

Train: 143,500 | Val: 30,750 | Test: 30,750

Class distribution check (train):
label
0.0    3500
0.3    3500
0.5    3500
0.8    3500
1.0    3500
1.3    3500
1.5    3500
1.8    3500
10     3500
2.0    3500
2.3    3500
2.5    3500
2.8    3500
3.0    3500
3.3    3500
3.5    3500
3.8    3500
4.0    3500
4.3    3500
4.5    3500
4.8    3500
5.0    3500
5.3    3500
5.5    3500
5.8    3500
6.0    3500
6.3    3500
6.5    3500
6.8    3500
7.0    3500
7.3    3500
7.5    3500
7.8    3500
8.0    3500
8.3    3500
8.5    3500
8.8    3500
9.0    3500
9.3    3500
9.5    3500
9.8    3500


# Perfectly balanced. Cell 3 – Verify no leakage:

In [3]:
# Kiểm tra không có ảnh nào xuất hiện ở 2 split khác nhau
train_paths = set(train_df['image_path'])
val_paths   = set(val_df['image_path'])
test_paths  = set(test_df['image_path'])

train_val_overlap  = train_paths & val_paths
train_test_overlap = train_paths & test_paths
val_test_overlap   = val_paths   & test_paths

print(f"Train∩Val overlap:  {len(train_val_overlap)}")
print(f"Train∩Test overlap: {len(train_test_overlap)}")
print(f"Val∩Test overlap:   {len(val_test_overlap)}")

if not any([train_val_overlap, train_test_overlap, val_test_overlap]):
    print("\n✅ No leakage detected. Splits are clean.")
else:
    print("\n❌ LEAKAGE DETECTED!")

Train∩Val overlap:  0
Train∩Test overlap: 0
Val∩Test overlap:   0

✅ No leakage detected. Splits are clean.
